[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Testing Failure


## What you will be able to do

Test what code does with bad input as well as good: check with `pytest.raises` that a call raises the
exception it should, check the message with `match` and the exception's own attributes with
`excinfo`, test a table of bad inputs at once, and check a warning with `pytest.warns`. Recognize a
test of an error that cannot fail, and fix it.


## The idea

### The problem

`parse_reading` has been tested with good lines, and a file of readings also brings bad ones: a line
with no comma, a word where a temperature should be, a station with no name. What the code does with
them matters as much as what it does with good lines. It should stop with an error that says what is
wrong and on which line, so that the person who reads the error can fix the file. That is a behavior,
and it needs tests like any other.

The **Why Test** notebook's solutions tested an error with `try`, `except` and `else`, as the **A
Real Client** notebook in the **APIs and JSON** guide did:

```python
def test_a_line_with_no_comma_raises_value_error():
    try:
        parse_reading("Bergen")
    except ValueError:
        pass
    else:
        raise AssertionError("parse_reading('Bergen') raised nothing")
```

Seven lines to state one expectation, and the `else` is the line people forget. Without it, the test
passes whether the error comes or not.

### What pytest.raises is

> **`pytest.raises`** is a context manager that expects an exception.
> `with pytest.raises(ReadingError):` runs the block inside it, and the test passes only if the
> block raises a `ReadingError` or a subclass of it. If the block raises nothing, the test fails
> with `DID NOT RAISE`, and if the block raises a different exception, that exception fails the test
> as it would anywhere else. **`match`** adds a regular expression that `re.search` must find in the
> exception's message, and `as excinfo` keeps an **`ExceptionInfo`**, whose `type` and `value` are
> the exception's class and the exception itself. **`pytest.warns`** does the same for a warning.

### Why it works that way

- **A context manager sees an exception leave its block.** The **Context Managers and Iterators**
  notebook in the **Object-Oriented Python** guide showed `__exit__` receiving the exception.
  `pytest.raises` checks the exception's type there, and either stops the exception or fails the
  test.
- **An error is part of a function's behavior.** A test of an error arranges bad input, acts, and
  asserts that the right thing went wrong.
- **The type says what kind of failure, and the message says which one.** `ReadingError` is any bad
  line, and `match="not a temperature"` is a bad temperature in particular.
- **A subclass counts.** `ReadingError` is a `ValueError`, so `pytest.raises(ValueError)` passes for
  it, and a test should name the narrowest type the code promises.
- **Only the raising line belongs in the block.** The block ends where the exception is raised, so a
  line after the raising call never runs.
- **A pattern is a regular expression.** Brackets, parentheses and dots in a message mean something
  to `re`, and `re.escape` turns a message into a pattern that matches only that message.

### Where you will meet this

pytest's documentation describes `pytest.raises` among its assertions about expected exceptions, and
notes that lines after the raising one, inside the block, do not run. The **Errors and Exceptions**
notebook in the **Python from the Start** guide defined an exception class of its own, and
`ReadingError`, in this notebook, is one. The **Regular Expressions** notebook in the same guide
explains the patterns `match` takes.

### What this notebook covers

- `pytest.raises`, in place of `try`, `except` and `else`
- A test that fails with `DID NOT RAISE`, and the bug it found
- `match`: the right exception, raised for the right reason
- `excinfo`: the exception's type, its message and its attributes
- A subclass that passes, and an exception of the wrong type that fails
- A table of bad lines, every one with its message, with `re.escape`
- `pytest.warns`, for a function that has a new name
- A file with a bad line, tested from the line's number to the script's exit code
- Four errors: an assert after the raising line, `pytest.raises(Exception)`, a pattern that does not
  match, and a warning that never comes

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
from pathlib import Path

Path("test_look.py").write_text("""
import pytest


def to_celsius(text):
    return float(text)


def test_a_word():
    with pytest.raises(ValueError, match="could not convert"):
        to_celsius("warm")


def test_nan():
    with pytest.raises(ValueError):
        to_celsius("nan")
""")

finished = subprocess.run([sys.executable, "-m", "pytest", "-q", "--tb=no"],
                          capture_output=True, text=True)
print(finished.stdout.strip())
```

```
.F                                                                       [100%]
=========================== short test summary info ============================
FAILED test_look.py::test_nan - Failed: DID NOT RAISE <class 'ValueError'>
1 failed, 1 passed in 0.01s
```

Both tests expect `ValueError`. The first passed: `float("warm")` raised it, with a message that
`match` found. The second failed with `DID NOT RAISE`, because `float` turns `"nan"` into the value
`nan`, which stands for not a number, and raises nothing. A test that expects an error can find a bug
by waiting for an error that never comes.


## Setup

Seven imports, and the functions that run pytest.

- `subprocess` runs pytest as a program of its own, in `run_pytest`
- `sys` names the Python that runs it, and puts the project's folder on the import path for the one
  cell that imports the module here
- `os` passes pytest the environment, with `NO_COLOR` and `PYTHONDONTWRITEBYTECODE` set in it, as in
  the **Your First Test** notebook
- `re` takes out of pytest's report the parts that differ between computers
- `Path` makes the project's folders
- `pytest` runs `pytest.raises` in that one cell, outside a test, to show what it keeps
- `shutil` removes the scratch folder at the end

`pytest_report` and `run_pytest` are the functions the **Your First Test** notebook wrote, which run
`python -m pytest` in the project's folder, `scratch/stations`, and that notebook explains each of
their settings.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import pytest

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("pytest", pytest.__version__, "| ready:", PROJECT)


pytest 8.4.2 | ready: scratch/stations


## Worked examples

### pytest.raises, in place of try, except and else

Until now, a bad line stopped `parse_reading` with whatever Python raised: `ValueError` from
unpacking, or from `float`. The module now raises an exception of its own, `ReadingError`, with a
message that says what is wrong with the line. `ReadingError` is a subclass of `ValueError`, so code
that already catches `ValueError` still catches it. `summarize` adds the line's number, and an old
name, `parse_line`, stays for code that still uses it:


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics
import warnings


class ReadingError(ValueError):
    """A line that is not a reading. line_number is the line's number in its file, when it is known."""

    def __init__(self, message, line_number=None):
        super().__init__(message)
        self.line_number = line_number


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    A line that is not a reading raises ReadingError.
    """
    fields = line.strip().split(",")
    if len(fields) != 2:
        raise ReadingError(f"expected 2 fields, got {len(fields)}: {fields}")
    station, celsius = fields
    if not station:
        raise ReadingError(f"no station name: {line.strip()!r}")
    celsius = celsius.replace("\u2212", "-")
    if not celsius:
        return station, None
    try:
        temperature = float(celsius)
    except ValueError:
        raise ReadingError(f"not a temperature: {celsius!r}") from None
    return station, temperature


def parse_line(line):
    """The old name of parse_reading, kept for code that still uses it."""
    warnings.warn("parse_line is deprecated; use parse_reading", DeprecationWarning, stacklevel=2)
    return parse_reading(line)


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped.

    A line that is not a reading raises ReadingError, with the line's number.
    """
    by_station = {}
    for number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            station, celsius = parse_reading(line)
        except ReadingError as error:
            raise ReadingError(f"line {number}: {error}", line_number=number) from error
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def summarize_file(path):
    """Each station's mean temperature, from a file of readings."""
    with open(path, encoding="utf-8-sig") as file:
        return summarize(file)


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


A test of an error puts the call that should raise inside `with pytest.raises(...)`, and does nothing
else:


In [3]:
%%writefile scratch/stations/tests/test_errors.py
import pytest

from readings import ReadingError, parse_reading


def test_a_line_with_no_comma_raises_reading_error():
    with pytest.raises(ReadingError):
        parse_reading("Bergen")


def test_a_word_is_not_a_temperature():
    with pytest.raises(ReadingError):
        parse_reading("Bergen,warm")


def test_a_line_with_no_station_raises_reading_error():
    with pytest.raises(ReadingError):
        parse_reading(",4.2")


Writing scratch/stations/tests/test_errors.py


In [4]:
run_pytest("tests/test_errors.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_errors.py::test_a_line_with_no_comma_raises_reading_error PASSED [ 33%]
tests/test_errors.py::test_a_word_is_not_a_temperature PASSED            [ 66%]
tests/test_errors.py::test_a_line_with_no_station_raises_reading_error PASSED [100%]

============================== 3 passed ===============================


Every test passed, because every call raised `ReadingError`, and `pytest.raises` stopped the
exception there, so that it never reached pytest as a failure. Two lines replace the seven of `try`,
`except` and `else`, and there is no `else` to forget: `pytest.raises` fails the test by itself when
nothing is raised.

### DID NOT RAISE: the error that never came

A reading of `nan` is not a temperature either, and a test says so:


In [5]:
%%writefile scratch/stations/tests/test_not_a_number.py
import pytest

from readings import ReadingError, parse_reading


def test_nan_is_not_a_temperature():
    with pytest.raises(ReadingError):
        parse_reading("Bergen,nan")


def test_infinity_is_not_a_temperature():
    with pytest.raises(ReadingError):
        parse_reading("Bergen,inf")


Writing scratch/stations/tests/test_not_a_number.py


In [6]:
run_pytest("tests/test_not_a_number.py", "-q")


FF                                                                       [100%]
=================================== FAILURES ===================================
________________________ test_nan_is_not_a_temperature _________________________

    def test_nan_is_not_a_temperature():
>       with pytest.raises(ReadingError):
E       Failed: DID NOT RAISE <class 'readings.ReadingError'>

tests/test_not_a_number.py:7: Failed
______________________ test_infinity_is_not_a_temperature ______________________

    def test_infinity_is_not_a_temperature():
>       with pytest.raises(ReadingError):
E       Failed: DID NOT RAISE <class 'readings.ReadingError'>

tests/test_not_a_number.py:12: Failed
=========================== short test summary info ============================
FAILED tests/test_not_a_number.py::test_nan_is_not_a_temperature - Failed: DI...
FAILED tests/test_not_a_number.py::test_infinity_is_not_a_temperature - Faile...
2 failed


`Failed: DID NOT RAISE`, and the class it waited for. `float` accepts `"nan"` and `"inf"`, and
returns the values that stand for not a number and for infinity, so `parse_reading` returned them
as temperatures, and a mean with a `nan` in it is `nan`. The tests found a bug by expecting an error
that never came. The fix checks that a temperature is a finite number, with `math.isfinite`:


In [7]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import math
import statistics
import warnings


class ReadingError(ValueError):
    """A line that is not a reading. line_number is the line's number in its file, when it is known."""

    def __init__(self, message, line_number=None):
        super().__init__(message)
        self.line_number = line_number


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    A line that is not a reading raises ReadingError.
    """
    fields = line.strip().split(",")
    if len(fields) != 2:
        raise ReadingError(f"expected 2 fields, got {len(fields)}: {fields}")
    station, celsius = fields
    if not station:
        raise ReadingError(f"no station name: {line.strip()!r}")
    celsius = celsius.replace("\u2212", "-")
    if not celsius:
        return station, None
    try:
        temperature = float(celsius)
    except ValueError:
        raise ReadingError(f"not a temperature: {celsius!r}") from None
    if not math.isfinite(temperature):
        raise ReadingError(f"not a temperature: {celsius!r}")
    return station, temperature


def parse_line(line):
    """The old name of parse_reading, kept for code that still uses it."""
    warnings.warn("parse_line is deprecated; use parse_reading", DeprecationWarning, stacklevel=2)
    return parse_reading(line)


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped.

    A line that is not a reading raises ReadingError, with the line's number.
    """
    by_station = {}
    for number, line in enumerate(lines, start=1):
        if not line.strip():
            continue
        try:
            station, celsius = parse_reading(line)
        except ReadingError as error:
            raise ReadingError(f"line {number}: {error}", line_number=number) from error
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def summarize_file(path):
    """Each station's mean temperature, from a file of readings."""
    with open(path, encoding="utf-8-sig") as file:
        return summarize(file)


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Overwriting scratch/stations/readings.py


In [8]:
run_pytest("tests", "-q")


.....                                                                    [100%]
5 passed


### match: the right exception, for the right reason

`pytest.raises(ReadingError)` passes for any bad line, whatever is wrong with it. `match` also checks
the message, with `re.search`, so a part of the message is enough. Here the second test expects a bad
temperature, and gets a line with three fields:


In [9]:
%%writefile scratch/stations/tests/test_messages.py
import pytest

from readings import ReadingError, parse_reading


def test_a_word_is_not_a_temperature():
    with pytest.raises(ReadingError, match="not a temperature"):
        parse_reading("Bergen,warm")


def test_a_word_after_a_comma_is_not_a_temperature():
    with pytest.raises(ReadingError, match="not a temperature"):
        parse_reading("Bergen,4.2,warm")


Writing scratch/stations/tests/test_messages.py


In [10]:
run_pytest("tests/test_messages.py", "-q")


.F                                                                       [100%]
=================================== FAILURES ===================================
________________ test_a_word_after_a_comma_is_not_a_temperature ________________

    def test_a_word_after_a_comma_is_not_a_temperature():
        with pytest.raises(ReadingError, match="not a temperature"):
>           parse_reading("Bergen,4.2,warm")

tests/test_messages.py:13: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = 'Bergen,4.2,warm'

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.
    
        A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
        A line that is not a reading raises ReadingError.
        """
        fields = line.strip().split(",")
        if len(fields) != 2:
>           raise ReadingError(f"expected 2 fields, got {len(fields)}: {fields}")

The second test did raise `ReadingError`, and for a different reason: its line has three fields, so
`parse_reading` never looked at a temperature. The report shows the exception that was raised, and
under it the pattern and the message it was matched against. Without `match`, this test would have
passed, and said nothing about the temperature it meant to test. The line that tests a temperature
has two fields:


In [11]:
source = (PROJECT / "tests" / "test_messages.py").read_text()
(PROJECT / "tests" / "test_messages.py").write_text(source.replace('"Bergen,4.2,warm"', '"Bergen,cold"'))

run_pytest("tests/test_messages.py", "-q")


..                                                                       [100%]
2 passed


### excinfo: the exception, kept

`as excinfo` keeps what `pytest.raises` caught, for asserts after the block. `pytest.raises` is an
ordinary context manager, so it also works in a notebook cell, which shows what it keeps:


In [12]:
sys.path.insert(0, str(PROJECT))
import readings

with pytest.raises(readings.ReadingError) as excinfo:
    readings.summarize(["Bergen,4.2", "", "Bergen,warm"])

print("type:        ", excinfo.type.__name__)
print("value:       ", excinfo.value)
print("line_number: ", excinfo.value.line_number)
print("its cause:   ", repr(excinfo.value.__cause__))


type:         ReadingError
value:        line 3: not a temperature: 'warm'
line_number:  3
its cause:    ReadingError("not a temperature: 'warm'")


`excinfo.type` is the exception's class and `excinfo.value` the exception itself, with the
attributes its class gave it, here the line's number. `summarize` raised its `ReadingError` with
`from error`, so the `ReadingError` that `parse_reading` raised is kept as its `__cause__`. A test
asserts on those, after the block:


In [13]:
%%writefile scratch/stations/tests/test_line_numbers.py
import pytest

from readings import ReadingError, summarize


def test_the_error_names_the_line_it_came_from():
    with pytest.raises(ReadingError) as excinfo:
        summarize(["Bergen,4.2", "", "Bergen,warm"])

    assert excinfo.value.line_number == 3
    assert str(excinfo.value) == "line 3: not a temperature: 'warm'"


Writing scratch/stations/tests/test_line_numbers.py


In [14]:
run_pytest("tests/test_line_numbers.py", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

tests/test_line_numbers.py::test_the_error_names_the_line_it_came_from PASSED [100%]

============================== 1 passed ===============================


The blank second line counts, since a person reading the file counts it too.

### A subclass passes, and a different exception fails

`pytest.raises(ValueError)` passes for a `ReadingError`, because a `ReadingError` is a `ValueError`.
`pytest.raises(ReadingError)` does not pass for a plain `ValueError`. Here `parse_reading` loses its
`try`, and `float`'s own `ValueError` escapes, while two tests expect each of the two classes:


In [15]:
%%writefile scratch/stations/tests/test_classes.py
import pytest

from readings import ReadingError, parse_reading


def test_a_word_raises_a_value_error():
    with pytest.raises(ValueError):
        parse_reading("Bergen,warm")


def test_a_word_raises_a_reading_error():
    with pytest.raises(ReadingError):
        parse_reading("Bergen,warm")


Writing scratch/stations/tests/test_classes.py


In [16]:
module = (PROJECT / "readings.py").read_text()          # the module as it is, to put back
without_try = module.replace(
    '    try:\n        temperature = float(celsius)\n    except ValueError:\n'
    '        raise ReadingError(f"not a temperature: {celsius!r}") from None\n',
    "    temperature = float(celsius)\n")
(PROJECT / "readings.py").write_text(without_try)

run_pytest("tests/test_classes.py", "-q")


.F                                                                       [100%]
=================================== FAILURES ===================================
______________________ test_a_word_raises_a_reading_error ______________________

    def test_a_word_raises_a_reading_error():
        with pytest.raises(ReadingError):
>           parse_reading("Bergen,warm")

tests/test_classes.py:13: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = 'Bergen,warm'

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.
    
        A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
        A line that is not a reading raises ReadingError.
        """
        fields = line.strip().split(",")
        if len(fields) != 2:
            raise ReadingError(f"expected 2 fields, got {len(fields)}: {fields}")
        station, celsius = fields
        if no

The test that expected `ValueError` still passed, since `float` raised one, and it could not tell
that the module had lost its own error. The test that expected `ReadingError` failed with the
`ValueError` itself: `pytest.raises` stops only the class it was given, and lets anything else
through. The narrower class tests the promise the module makes. The `try` goes back:


In [17]:
(PROJECT / "readings.py").write_text(module)

run_pytest("tests/test_classes.py", "-q")


..                                                                       [100%]
2 passed


### A table of bad lines

The **Parametrize** notebook's tables work for errors too. Every bad line gets its whole message, and
`re.escape` makes the message a pattern that matches only itself, since a message such as
`['Bergen', '4.2', 'extra']` is full of characters that mean something to a regular expression. `^`
and `$` make the pattern match the whole message, and not merely a part:


In [18]:
%%writefile scratch/stations/tests/test_bad_lines.py
import re

import pytest

from readings import ReadingError, parse_reading

BAD_LINES = [
    pytest.param("Bergen", "expected 2 fields, got 1: ['Bergen']", id="no comma"),
    pytest.param("Bergen,4.2,extra", "expected 2 fields, got 3: ['Bergen', '4.2', 'extra']", id="an extra field"),
    pytest.param(",4.2", "no station name: ',4.2'", id="no station"),
    pytest.param("Bergen,warm", "not a temperature: 'warm'", id="a word"),
    pytest.param("Bergen,nan", "not a temperature: 'nan'", id="not a number"),
    pytest.param("Bergen,inf", "not a temperature: 'inf'", id="infinity"),
]


@pytest.mark.parametrize("line, message", BAD_LINES)
def test_a_bad_line_raises_reading_error(line, message):
    with pytest.raises(ReadingError, match=f"^{re.escape(message)}$"):
        parse_reading(line)


Writing scratch/stations/tests/test_bad_lines.py


In [19]:
run_pytest("tests/test_bad_lines.py", "-v")


============================= test session starts ==============================
collecting ... collected 6 items

tests/test_bad_lines.py::test_a_bad_line_raises_reading_error[no comma] PASSED [ 16%]
tests/test_bad_lines.py::test_a_bad_line_raises_reading_error[an extra field] PASSED [ 33%]
tests/test_bad_lines.py::test_a_bad_line_raises_reading_error[no station] PASSED [ 50%]
tests/test_bad_lines.py::test_a_bad_line_raises_reading_error[a word] PASSED [ 66%]
tests/test_bad_lines.py::test_a_bad_line_raises_reading_error[not a number] PASSED [ 83%]
tests/test_bad_lines.py::test_a_bad_line_raises_reading_error[infinity] PASSED [100%]

============================== 6 passed ===============================


Six kinds of bad line, one test, and a message checked for every one. A new kind of bad line is one
more row.

### pytest.warns: a function with a new name

`parse_line` still works, and warns that it is deprecated, so that code using it learns to move to
`parse_reading` before the old name goes. A test that calls it without expecting the warning passes,
and pytest collects the warning into the report:


In [20]:
%%writefile scratch/stations/tests/test_old_name.py
from readings import parse_line


def test_the_old_name_still_parses():
    assert parse_line("Bergen,4.2") == ("Bergen", 4.2)


Writing scratch/stations/tests/test_old_name.py


In [21]:
run_pytest("tests/test_old_name.py", "-q")


.                                                                        [100%]
=============================== warnings summary ===============================
tests/test_old_name.py::test_the_old_name_still_parses
  tests/test_old_name.py:5: DeprecationWarning: parse_line is deprecated; use parse_reading
    assert parse_line("Bergen,4.2") == ("Bergen", 4.2)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning


The warning names the test's line, since `parse_line` passes `stacklevel=2` to `warnings.warn`, which
points the warning at the code that called it. `pytest.warns` makes the warning part of what the test
checks, with `match` for its message, and it keeps the value the call returned for an assert after
the block:


In [22]:
%%writefile scratch/stations/tests/test_old_name.py
import pytest

from readings import parse_line


def test_the_old_name_warns_about_the_new_one():
    with pytest.warns(DeprecationWarning, match="use parse_reading"):
        parse_line("Bergen,4.2")


def test_the_old_name_still_parses():
    with pytest.warns(DeprecationWarning):
        result = parse_line("Bergen,4.2")

    assert result == ("Bergen", 4.2)


Overwriting scratch/stations/tests/test_old_name.py


In [23]:
run_pytest("tests/test_old_name.py", "-q")


..                                                                       [100%]
2 passed


No warnings in the report now: both tests expected theirs.

### A file with a bad line, from the line to the exit code

The pieces of this notebook, in one file of tests. The summary script now stops at a bad line with a
message that names the file and the line, and with `sys.exit`, which raises `SystemExit` and ends the
program with exit code 1:


In [24]:
%%writefile scratch/stations/summary.py
"""Print each station's mean temperature from a file of readings.

    python summary.py readings.csv

With no path given, the path comes from the environment variable STATIONS_READINGS. A line that is
not a reading stops the script, with a message that names the file and the line.
"""

import os
import sys

from readings import ReadingError, summarize_file, to_fahrenheit


def main(arguments):
    path = arguments[0] if arguments else os.environ["STATIONS_READINGS"]
    try:
        summary = summarize_file(path)
    except ReadingError as error:
        sys.exit(f"{path}: {error}")
    for station, celsius in summary.items():
        if celsius is None:
            print(station, "no readings")
        else:
            print(station, f"{celsius:.1f} C, {to_fahrenheit(celsius):.1f} F")


if __name__ == "__main__":
    main(sys.argv[1:])


Writing scratch/stations/summary.py


In [25]:
%%writefile scratch/stations/tests/test_bad_files.py
import subprocess
import sys

import pytest

from readings import ReadingError, summarize_file
from summary import main


@pytest.fixture
def bad_file(tmp_path):
    """A file whose second line holds a word where a temperature should be."""
    path = tmp_path / "bad.csv"
    path.write_text("Bergen,4.2\nBergen,warm\nOslo,-2.4\n", encoding="utf-8")
    return path


def test_the_module_names_the_line(bad_file):
    with pytest.raises(ReadingError, match=r"^line 2: not a temperature: 'warm'$") as excinfo:
        summarize_file(bad_file)

    assert excinfo.value.line_number == 2


def test_main_exits_with_the_file_and_the_line(bad_file):
    with pytest.raises(SystemExit) as excinfo:
        main([str(bad_file)])

    assert excinfo.value.code == f"{bad_file}: line 2: not a temperature: 'warm'"


def test_the_program_ends_with_exit_code_1(bad_file):
    finished = subprocess.run([sys.executable, "summary.py", str(bad_file)], capture_output=True, text=True)

    assert finished.returncode == 1
    assert finished.stderr.strip() == f"{bad_file}: line 2: not a temperature: 'warm'"


Writing scratch/stations/tests/test_bad_files.py


In [26]:
run_pytest("tests/test_bad_files.py", "-v")
print()
run_pytest("-q")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_bad_files.py::test_the_module_names_the_line PASSED           [ 33%]
tests/test_bad_files.py::test_main_exits_with_the_file_and_the_line PASSED [ 66%]
tests/test_bad_files.py::test_the_program_ends_with_exit_code_1 PASSED   [100%]

============================== 3 passed ===============================

.....................                                                    [100%]
21 passed


Three tests of one bad line, from three distances. The first checks the module's exception and its
line number. The second catches the `SystemExit` that `sys.exit` raises, whose `code` is the
message, without ending the test. The third starts the program, and checks the exit code and what it
printed to standard error, which is where `sys.exit` prints a message.

### Where each part came from

| In the tests | What it relies on | The section that showed it |
|---|---|---|
| `with pytest.raises(ReadingError, ...)` | a test that passes only when the block raises | pytest.raises, in place of try, except and else |
| `match=r"^line 2: ...$"` | the message checked, with a pattern that matches all of it | match: the right exception, for the right reason |
| `excinfo.value.line_number` | the exception, kept for asserts after the block | excinfo: the exception, kept |
| `ReadingError`, not `ValueError` | the narrowest class the code promises | A subclass passes, and a different exception fails |
| `pytest.raises(SystemExit)` | any exception, `SystemExit` included, can be expected | pytest.raises, in place of try, except and else |
| nan and inf rejected by the module | a test that failed with `DID NOT RAISE` | DID NOT RAISE: the error that never came |

A file with a bad line now stops the summary with a message that says where to look, and every step
of that is tested.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/06-testing-failure-solutions.ipynb).

**1.** Write `tests/test_tasks.py` with a test that checks that `parse_reading("Oslo")` raises
`ReadingError`. Run the file.


In [27]:
# your code here


**2.** Add a test that checks that `parse_reading("Oslo,cold")` raises `ReadingError` with a message
containing `not a temperature: 'cold'`. Run the file.


In [28]:
# your code here


**3.** Add a test that checks that `parse_reading("Oslo,-2.4,extra")` raises `ValueError`. Run the
file, and say why it passes although the module raises `ReadingError`.


In [29]:
# your code here


**4.** Add a test that uses `excinfo` to check that `summarize(["Oslo,-2.4", "", "Oslo,cold"])`
raises a `ReadingError` whose `line_number` is 3. Run the file.


In [30]:
# your code here


**5.** Write `tests/test_temperatures.py` with a test parametrized over three bad temperatures,
`"cold"`, `"inf"` and `"-"`, with IDs, which checks that a line from Oslo with each of them raises
`ReadingError` with the message `not a temperature: ` and the temperature in quotes. Run it with
`-v`.


In [31]:
# your code here


**6.** Write a test that checks that `parse_line("Oslo,-2.4")` warns with a `DeprecationWarning` and
returns `("Oslo", -2.4)`. Run it.


In [32]:
# your code here


## Common errors

### No error, and a check that never ran: an assert after the raising line


In [33]:
%%writefile scratch/stations/tests/test_after.py
import pytest

from readings import ReadingError, summarize


def test_the_error_names_the_line():
    with pytest.raises(ReadingError) as excinfo:
        summarize(["Bergen,4.2", "Bergen,warm"])
        assert excinfo.value.line_number == 99


Writing scratch/stations/tests/test_after.py


In [34]:
run_pytest("tests/test_after.py", "-q")


.                                                                        [100%]
1 passed


The test passed, and the line number is 2, not 99. `summarize` raised, the block ended there, and the
`assert` inside the block never ran, as pytest's documentation warns. An `assert` about the exception
belongs after the block, where it runs:


In [35]:
source = (PROJECT / "tests" / "test_after.py").read_text()
(PROJECT / "tests" / "test_after.py").write_text(source.replace("\n        assert", "\n\n    assert"))

run_pytest("tests/test_after.py", "-q")


F                                                                        [100%]
=================================== FAILURES ===================================
________________________ test_the_error_names_the_line _________________________

    def test_the_error_names_the_line():
        with pytest.raises(ReadingError) as excinfo:
            summarize(["Bergen,4.2", "Bergen,warm"])
    
>       assert excinfo.value.line_number == 99
E       assert 2 == 99
E        +  where 2 = ReadingError("line 2: not a temperature: 'warm'").line_number
E        +    where ReadingError("line 2: not a temperature: 'warm'") = <ExceptionInfo ReadingError("line 2: not a temperature: 'warm'") tblen=2>.value

tests/test_after.py:10: AssertionError
=========================== short test summary info ============================
FAILED tests/test_after.py::test_the_error_names_the_line - assert 2 == 99
1 failed


Now the `assert` runs, and fails, which shows it checks something. The expected number is the one to
correct:


In [36]:
source = (PROJECT / "tests" / "test_after.py").read_text()
(PROJECT / "tests" / "test_after.py").write_text(source.replace("== 99", "== 2"))

run_pytest("tests/test_after.py", "-q")


.                                                                        [100%]
1 passed


### No error, and the wrong failure accepted: pytest.raises(Exception)


In [37]:
%%writefile scratch/stations/tests/test_broad.py
import pytest

from readings import parse_reading


def test_a_bad_line_raises():
    with pytest.raises(Exception):
        parse_readng("Bergen")


Writing scratch/stations/tests/test_broad.py


In [38]:
run_pytest("tests/test_broad.py", "-q")


.                                                                        [100%]
1 passed


The test passed, and `parse_reading` never ran: the name is misspelled, so the block raised
`NameError`, and `Exception` accepts that as readily as `ReadingError`. A class wide enough to catch
anything cannot tell the error the test is about from a mistake in the test. Name the class the code
promises, and the typo shows itself:


In [39]:
source = (PROJECT / "tests" / "test_broad.py").read_text()
(PROJECT / "tests" / "test_broad.py").write_text(source.replace("import parse_reading", "import ReadingError, parse_reading")
                                                       .replace("pytest.raises(Exception)", "pytest.raises(ReadingError)"))

run_pytest("tests/test_broad.py", "-q", "--tb=no")


F                                                                        [100%]
=========================== short test summary info ============================
FAILED tests/test_broad.py::test_a_bad_line_raises - NameError: name 'parse_r...
1 failed


In [40]:
source = (PROJECT / "tests" / "test_broad.py").read_text()
(PROJECT / "tests" / "test_broad.py").write_text(source.replace("parse_readng(", "parse_reading("))

run_pytest("tests/test_broad.py", "-q")


.                                                                        [100%]
1 passed


### AssertionError: Regex pattern did not match.


In [41]:
%%writefile scratch/stations/tests/test_pattern.py
import pytest

from readings import ReadingError, parse_reading


def test_an_extra_field_is_named():
    with pytest.raises(ReadingError, match="got 3: ['Bergen', '4.2', 'extra']"):
        parse_reading("Bergen,4.2,extra")


Writing scratch/stations/tests/test_pattern.py


In [42]:
run_pytest("tests/test_pattern.py", "-q")


F                                                                        [100%]
=================================== FAILURES ===================================
_________________________ test_an_extra_field_is_named _________________________

    def test_an_extra_field_is_named():
        with pytest.raises(ReadingError, match="got 3: ['Bergen', '4.2', 'extra']"):
>           parse_reading("Bergen,4.2,extra")

tests/test_pattern.py:8: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = 'Bergen,4.2,extra'

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.
    
        A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
        A line that is not a reading raises ReadingError.
        """
        fields = line.strip().split(",")
        if len(fields) != 2:
>           raise ReadingError(f"expected 2 fields, got {len(fields)}: {fields}")
E

The message contains the pattern's text exactly, and the pattern did not match. In a regular
expression, `[` starts a set of characters, so `['Bergen', '4.2', 'extra']` matches one character
from that set, not the text in brackets. `re.escape` puts a backslash before every character that
could mean something in a pattern, spaces included, as its output shows:


In [43]:
%%writefile scratch/stations/tests/test_pattern.py
import re

import pytest

from readings import ReadingError, parse_reading


def test_an_extra_field_is_named():
    with pytest.raises(ReadingError, match=re.escape("got 3: ['Bergen', '4.2', 'extra']")):
        parse_reading("Bergen,4.2,extra")


Overwriting scratch/stations/tests/test_pattern.py


In [44]:
print(re.escape("got 3: ['Bergen', '4.2', 'extra']"))

run_pytest("tests/test_pattern.py", "-q")


got\ 3:\ \['Bergen',\ '4\.2',\ 'extra'\]
.                                                                        [100%]
1 passed


### Failed: DID NOT WARN. No warnings of type (<class 'DeprecationWarning'>,) were emitted.


In [45]:
%%writefile scratch/stations/tests/test_warning.py
import pytest

from readings import parse_reading


def test_the_old_name_warns():
    with pytest.warns(DeprecationWarning):
        parse_reading("Bergen,4.2")


Writing scratch/stations/tests/test_warning.py


In [46]:
run_pytest("tests/test_warning.py", "-q")


F                                                                        [100%]
=================================== FAILURES ===================================
___________________________ test_the_old_name_warns ____________________________

    def test_the_old_name_warns():
>       with pytest.warns(DeprecationWarning):
E       Failed: DID NOT WARN. No warnings of type (<class 'DeprecationWarning'>,) were emitted.
E        Emitted warnings: [].

tests/test_warning.py:7: Failed
=========================== short test summary info ============================
FAILED tests/test_warning.py::test_the_old_name_warns - Failed: DID NOT WARN....
1 failed


`parse_reading` is the new name, and it has nothing to warn about, so `pytest.warns` saw no warning,
and its report lists the warnings it did see: none. The test meant the old name:


In [47]:
source = (PROJECT / "tests" / "test_warning.py").read_text()
(PROJECT / "tests" / "test_warning.py").write_text(source.replace("parse_reading", "parse_line"))

run_pytest("tests/test_warning.py", "-q")


.                                                                        [100%]
1 passed


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
module, the script, and every test file:


In [48]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `with pytest.raises(SomeError):` passes only if its block raises that class or a subclass, fails
  with `DID NOT RAISE` if the block raises nothing, and lets any other exception fail the test.
- `match` checks the message with `re.search`. Use `re.escape` for a message with special characters,
  and `^` and `$` to match all of it.
- `as excinfo` keeps the exception: `excinfo.type`, `excinfo.value`, and the attributes on
  `excinfo.value`, checked after the block.
- Put only the raising call inside the block, since nothing after it runs.
- Name the narrowest class the code promises, never `Exception`, which also accepts a mistake in the
  test.
- `pytest.warns` expects a warning in the same way, and `pytest.raises(SystemExit)` tests `sys.exit`.


## What is next

The **Virtual Environments** notebook turns to the other half of this guide, the place the tests run:
a folder with its own Python and its own packages, and a tutorial that breaks without one.


---

&#8592; **Previous:** [Parametrize](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/05-parametrize.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
